In [1]:
import sys
import os
import numpy as np 
import pandas as pd 
from torch.utils.data import DataLoader  
import yaml
from sr_model import paired_sr, single_sr
from dataset import omic_data,single_data
import torch
import muon as mu 
import json 
import scanpy as sc 

'''
train a paired model without pretraining weights
'''
def load_data():
    mdata = mu.read_h5mu('/home/rsun@ZHANGroup.local/multi_pretrain/evaluation/notebook/eval_data/M_rna_1/mdata.h5mu')
    
    rna = mdata['rna_count']
    gadata = mdata['ga_count'] 

    mutual_gene = np.load('/home/rsun@ZHANGroup.local/atac_pretrain/src/mutual_gene.npy', allow_pickle=True)
    gadata = gadata[:,mutual_gene]
    
    # lo1p transform 

    sc.pp.normalize_total(gadata, target_sum= 1e4)
    sc.pp.log1p(gadata)

    sc.pp.normalize_total(rna, target_sum= 1e4)
    sc.pp.log1p(rna)

    # feature selection

    sc.pp.highly_variable_genes(rna, n_top_genes= 5000)
    rna = rna[:,rna.var['highly_variable']]

    sc.pp.highly_variable_genes(gadata, n_top_genes= 10000)
    gadata = gadata[:,gadata.var['highly_variable']] 

    print(rna.shape, gadata.shape)

    #train_idx, test_idx = train_test_split(np.arange(N), test_size = 0.1, random_state = 42)
    train_idx = np.load('/home/rsun@ZHANGroup.local/multi_pretrain/evaluation/sr_result/train_test_split/train_id_1.npy', allow_pickle = True)
    test_idx = np.load('/home/rsun@ZHANGroup.local/multi_pretrain/evaluation/sr_result/train_test_split/test_id_1.npy', allow_pickle = True)

    rna_train, rna_test = rna[train_idx,:], rna[test_idx,:]
    gadata_train, gadata_test = gadata[train_idx,:], gadata[test_idx,:]
    return rna_train, rna_test, gadata_train, gadata_test

def process_data(rna_train, rna_test, gadata_train, gadata_test):

    train_rna = rna_train.X.toarray().astype(np.float32)
    train_ga = gadata_train.X.toarray().astype(np.float32)
    train_rna = torch.from_numpy(train_rna)
    train_ga = torch.from_numpy(train_ga)

    test_rna = rna_test.X.toarray().astype(np.float32)
    test_ga = gadata_test.X.toarray().astype(np.float32)
    test_rna = torch.from_numpy(test_rna)
    test_ga = torch.from_numpy(test_ga)
    return train_rna, train_ga, test_rna, test_ga

def set_rna_config(N, B=1024):
    # N is the dataset size , B is the batch size 

    if N >= 100000:
        print('Large dataset, please use pair_train.py')
        return None 
    steps = int(40*N/B) # at least 1000 steps 


    config = {
        'omic': {'model_type': 'rna'},

        'network': {
            'feature_num': 5000,
            'hidden_dims': [512, 256, 128],
            'dropout': 0.1,
            'layernorm_eps': 1e-8,
            'activation': 'leaky_relu',
            'input_dropout': 0.2,
            'vae_weight': 1,
            'ce_weights': None,
            'class_dict': None
        },
        'optimizer': {
            'learning_rate': 1e-4,
            'weight_decay': 0.01,
            'warmup_steps': 100,
            'anneal_steps': steps,
            'min_lr': 1e-6
        },
        'training': {
            'device': 'cuda',
            'training_steps': steps,
            'eval_steps': 100000,
            'save_steps': 100000, # set large, do not save checkpoint during training
            'log_dir': 'logs',
            'save_dir': 'saved_models',
            'run_name': 'tiny_rna'
        }
    }
    return config

def set_ga_config(N, B=1024):
    # N is the dataset size , B is the batch size 

    if N >= 100000:
        print('Large dataset, please use pair_train.py')
        return None 
    steps = int(60*N/B) # at least 1000 steps 


    config = {
        'omic': {'model_type': 'ga'},

        'network': {
            'feature_num': 10000,
            'hidden_dims': [512, 256, 128],
            'dropout': 0.1,
            'layernorm_eps': 1e-8,
            'activation': 'leaky_relu',
            'input_dropout': 0.2,
            'vae_weight': 1,
            'ce_weights': None,
            'class_dict': None
        },
        'optimizer': {
            'learning_rate': 1e-4,
            'weight_decay': 0.01,
            'warmup_steps': 100,
            'anneal_steps': steps,
            'min_lr': 1e-6
        },
        'training': {
            'device': 'cuda',
            'training_steps': steps,
            'eval_steps': 100000,
            'save_steps': 100000, # set large, do not save checkpoint during training
            'log_dir': 'logs',
            'save_dir': 'saved_models',
            'run_name': 'tiny_ga'
        }
    }
    return config
def rna_train(rna_data):
    print(1)
    rna_dataset = single_data(rna_data)
    print(rna_dataset)
    rna_loader = DataLoader(rna_dataset, batch_size = 1024, shuffle= True)
    for batch in rna_loader:
        print(batch)
        break

    '''
    ini config
    '''
    config = set_rna_config(N = rna_data.shape[0], B = 1024)
    print(config)

    '''
    set rna model 
    '''
    rna_model = single_sr(config)
    #rna_model.set_optimizer()
    rna_model.train_model(train_loader = rna_loader,save_config= False)
    return rna_model, config

def ga_train(ga_data):
    ga_dataset = single_data(ga_data)
    ga_loader = DataLoader(ga_dataset, batch_size = 1024, shuffle= True)

    '''
    ini config
    '''
    config = set_ga_config(N = ga_data.shape[0], B = 1024)

    '''
    set rna model 
    '''
    ga_model = single_sr(config)
    ga_model.set_optimizer()
    ga_model.train_model(train_loader = ga_loader)
    return ga_model, config 

def paired_train(paired_config,
                 rna_config,
                 ga_config,
                 sr_rna, 
                 sr_ga,
                 train_loader, 
                 val_loader):

    paired_model = paired_sr(paired_config, 
                             rna_config, 
                             ga_config,
                             sr_rna,
                             sr_ga)
    paired_model.train_model(train_loader,
                            val_loader)
    return paired_model


2025-02-24 14:21:01 - INFO - PyTorch version 2.5.1 available.
2025-02-24 14:21:01 - INFO - Polars version 1.16.0 available.
2025-02-24 14:21:01 - INFO - JAX version 0.4.35 available.


In [2]:
with open('/home/rsun@ZHANGroup.local/sr_project/configs/paired_configs/config_scratch.yaml', 'r') as file:
    config = yaml.safe_load(file)
print(config)
print('config over')

rna_train, rna_test, gadata_train, gadata_test = load_data()
train_rna, train_ga, test_rna, test_ga = process_data(rna_train, rna_test, gadata_train, gadata_test)
print(train_rna.shape, train_ga.shape)
print(test_rna.shape, test_ga.shape)
"""
prepare single omic dataset, for single omic data, we use all data to train the single omic model
"""

combined_rna = torch.cat((train_rna, test_rna), dim=0)
combined_ga = torch.cat((train_ga, test_ga), dim=0)
print(combined_rna.shape, combined_ga.shape)

#rna_dataset = single_data(combined_rna)
#ga_dataset = single_data(combined_ga)

"""
prepare  multiomic data
"""
traindata = omic_data(train_rna, train_ga)
testdata = omic_data(test_rna, test_ga) 
print('dataset over')


{'rna_config_path': None, 'ga_config_path': None, 'rna_checkpoint': None, 'ga_checkpoint': None, 'temperature': 0.1, 'tau': 0.5, 'freeze_param': False, 'learning_rate': 0.0001, 'weight_decay': 0.05, 'warmup_steps': 2000, 'anneal_steps': 20000, 'min_lr': 5e-06, 'log_dir': 'logs', 'save_dir': 'saved_models', 'run_name': 'paired_config_scratch', 'num_steps': 25000, 'device': 'cuda', 'eval_steps': 50, 'save_steps': 2000, 'batchsize': 4096}
config over


/home/rsun@ZHANGroup.local/anaconda3/envs/snapatac/lib/python3.10/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/rsun@ZHANGroup.local/anaconda3/envs/snapatac/lib/python3.10/site-packages/mudata/_core/mudata.py:931: UserWarning: Cannot join columns with the same name because var_names are intersecting.
  warnings.warn(
/home/rsun@ZHANGroup.local/anaconda3/envs/snapatac/lib/python3.10/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. U

(71002, 5000) (71002, 10000)
torch.Size([63901, 5000]) torch.Size([63901, 10000])
torch.Size([7101, 5000]) torch.Size([7101, 10000])
torch.Size([71002, 5000]) torch.Size([71002, 10000])
dataset over


In [5]:
combined_rna

torch.Size([71002, 5000])

In [3]:
sr_rna, rna_config = rna_train(combined_rna)

TypeError: 'AnnData' object is not callable

In [14]:
config = set_ga_config(16666)

In [5]:
ga_dataset = single_data(combined_ga)
ga_loader = DataLoader(ga_dataset, batch_size = 1024, shuffle= True)

In [6]:
for batch in ga_loader:
    break

In [8]:
batch['feature']

tensor([[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 1.4616],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]])